In [33]:
import pandas as pd

In [34]:
df = pd.read_csv('df.tsv', sep='\t', encoding='utf-8')  

In [51]:
df.columns

Index(['Unnamed: 0', 'glottocode', 'language_name', 'isocodes', 'level',
       'macroarea', 'latitude', 'longitude', 'dictionary', 'root_node',
       'bible', 'cc_GB', 'archival_GB', 'datasets_n', 'n_wiki_articles',
       'OPUS', 'name_lower', 'XEUS', 'wikitongues', 'model_n',
       'mean_hf_downloads', 'earliest_datetime', 'wiki_active_users',
       'institutional_status', 'endangerment_status', 'primary_country',
       'admin_1_code', 'country_iso_code', 'countries_where_spoken',
       'contested area?', 'n_speakers', 'geometry', 'nearest_uni',
       'second_nearest_uni', 'distance_to_nearest_km',
       'distance_to_second_nearest_km', 'hdi', 'gdp',
       'Cybersecurity legislation/regulation exist', 'households with phones',
       'households with home internet', 'individuals_using_internet',
       'avg_d_kbps', 'avg_u_kbps', 'avg_lat_ms', '% Stem Grads',
       'literacy rate', 'educational_attainment_score', 'RnD % of GDP',
       'phones'],
      dtype='object')

In [36]:
df_china = pd.read_excel('data/df_China.xlsx')

In [37]:
common_cols = [col for col in df.columns if col in df_china.columns and col != 'isocodes']

# 2. Set 'isocode' as index in both dataframes temporarily for easy alignment
df_indexed = df.set_index('isocodes')
df_china_indexed = df_china.set_index('isocodes')

# 3. Update df_indexed with values from df_china_indexed for the common columns
df_indexed.update(df_china_indexed[common_cols])

# 4. Reset index to bring back 'isocode' as a column
df_updated = df_indexed.reset_index()

# 5. If you want to update your original df in place:
df.update(df_updated)

In [39]:
wikidata_speakers = pd.read_json('data/wikidata_result.json')

In [41]:
wikidata_speakers['num_of_speaker']

0      {'name': 'Old Swedish', 'aliases': None, 'num_...
1      {'name': 'Afar', 'aliases': None, 'num_of_spea...
2      {'name': 'Abkhaz', 'aliases': None, 'num_of_sp...
3      {'name': 'Abenaki', 'aliases': None, 'num_of_s...
4      {'name': 'Abaza', 'aliases': None, 'num_of_spe...
                             ...                        
800    {'name': 'Cantonese', 'aliases': None, 'num_of...
801    {'name': 'Standard Malay', 'aliases': None, 'n...
802    {'name': 'Zulu', 'aliases': None, 'num_of_spea...
803    {'name': 'Zuni', 'aliases': None, 'num_of_spea...
804    {'name': 'no linguistic content', 'aliases': [...
Name: num_of_speaker, Length: 805, dtype: object

In [42]:
wikidata_speakers_expanded = wikidata_speakers['num_of_speaker'].apply(pd.Series)

In [43]:
wikidata_speakers_expanded

,name,aliases,num_of_speaker
0,Old Swedish,None,[]
1,Afar,None,"[{'number_of_speaker': '+1600000', 'type_of_sp..."
2,Abkhaz,None,"[{'number_of_speaker': '+190110', 'type_of_spe..."
3,Abenaki,None,"[{'number_of_speaker': '+20', 'type_of_speaker..."
4,Abaza,None,"[{'number_of_speaker': '+48000', 'type_of_spea..."
...,...,...,...
800,Cantonese,None,"[{'number_of_speaker': '+72893210', 'type_of_s..."
801,Standard Malay,None,[]
802,Zulu,None,"[{'number_of_speaker': '+12100000', 'type_of_s..."
803,Zuni,None,"[{'number_of_speaker': '+9650', 'type_of_speak..."


In [44]:
wikidata_speakers = wikidata_speakers.drop(columns=['num_of_speaker']).join(wikidata_speakers_expanded)

In [45]:
wikidata_speakers

,qid,language_code,language_name,name,aliases,num_of_speaker
0,Q2417210,http://www.wikidata.org/.well-known/genid/4622...,Old Swedish,Old Swedish,None,[]
1,Q27811,aa,Afar,Afar,None,"[{'number_of_speaker': '+1600000', 'type_of_sp..."
2,Q5111,ab,Abkhaz,Abkhaz,None,"[{'number_of_speaker': '+190110', 'type_of_spe..."
3,Q197936,abe,Abenaki,Abenaki,None,"[{'number_of_speaker': '+20', 'type_of_speaker..."
4,Q27567,abq,Abaza,Abaza,None,"[{'number_of_speaker': '+48000', 'type_of_spea..."
...,...,...,...,...,...,...
800,Q9186,zh-yue,Cantonese,Cantonese,None,"[{'number_of_speaker': '+72893210', 'type_of_s..."
801,Q123474569,zsm,Standard Malay,Standard Malay,None,[]
802,Q10179,zu,Zulu,Zulu,None,"[{'number_of_speaker': '+12100000', 'type_of_s..."
803,Q10188,zun,Zuni,Zuni,None,"[{'number_of_speaker': '+9650', 'type_of_speak..."


In [47]:
# Step 1: Explode the list of dictionaries so each dictionary becomes its own row
wikidata_speakers_exploded = wikidata_speakers.explode('num_of_speaker').reset_index(drop=True)

# Step 2: Expand the dictionaries in 'num_of_speakers' column into separate columns
num_speakers_expanded = wikidata_speakers_exploded['num_of_speaker'].apply(pd.Series)

# Step 3: Join the expanded columns back to the exploded dataframe (dropping original list column if you want)
df_final = wikidata_speakers_exploded.drop(columns=['num_of_speaker']).join(num_speakers_expanded)

In [48]:
df_final

,qid,language_code,language_name,name,aliases,0,number_of_speaker,type_of_speaker,data_time
0,Q2417210,http://www.wikidata.org/.well-known/genid/4622...,Old Swedish,Old Swedish,None,NaN,NaN,NaN,NaN
1,Q27811,aa,Afar,Afar,None,NaN,+1600000,None,None
2,Q5111,ab,Abkhaz,Abkhaz,None,NaN,+190110,None,+2015-00-00T00:00:00Z
3,Q197936,abe,Abenaki,Abenaki,None,NaN,+20,None,+2012-00-00T00:00:00Z
4,Q197936,abe,Abenaki,Abenaki,None,NaN,+20,None,+1991-00-00T00:00:00Z
...,...,...,...,...,...,...,...,...,...
1101,Q10179,zu,Zulu,Zulu,None,NaN,+12100000,Q36870,+2019-00-00T00:00:00Z
1102,Q10179,zu,Zulu,Zulu,None,NaN,+11969100,Q36870,+2011-00-00T00:00:00Z
1103,Q10179,zu,Zulu,Zulu,None,NaN,+15700000,Q125421,+2011-00-00T00:00:00Z
1104,Q10188,zun,Zuni,Zuni,None,NaN,+9650,None,None


In [49]:
df_final['number_of_speaker'] = df_final['number_of_speaker'].str.lstrip('+')

In [50]:
df_final

,qid,language_code,language_name,name,aliases,0,number_of_speaker,type_of_speaker,data_time
0,Q2417210,http://www.wikidata.org/.well-known/genid/4622...,Old Swedish,Old Swedish,None,NaN,NaN,NaN,NaN
1,Q27811,aa,Afar,Afar,None,NaN,1600000,None,None
2,Q5111,ab,Abkhaz,Abkhaz,None,NaN,190110,None,+2015-00-00T00:00:00Z
3,Q197936,abe,Abenaki,Abenaki,None,NaN,20,None,+2012-00-00T00:00:00Z
4,Q197936,abe,Abenaki,Abenaki,None,NaN,20,None,+1991-00-00T00:00:00Z
...,...,...,...,...,...,...,...,...,...
1101,Q10179,zu,Zulu,Zulu,None,NaN,12100000,Q36870,+2019-00-00T00:00:00Z
1102,Q10179,zu,Zulu,Zulu,None,NaN,11969100,Q36870,+2011-00-00T00:00:00Z
1103,Q10179,zu,Zulu,Zulu,None,NaN,15700000,Q125421,+2011-00-00T00:00:00Z
1104,Q10188,zun,Zuni,Zuni,None,NaN,9650,None,None


In [54]:
code_to_speakers = dict(zip(df_final['language_code'], df_final['number_of_speaker']))
name_to_speakers = dict(zip(df_final['language_name'], df_final['number_of_speaker']))

# Map 'isocodes' to number_of_speaker where possible
mapped_by_code = df['isocodes'].map(code_to_speakers)

# Map 'language_name' to number_of_speaker where possible
mapped_by_name = df['language_name'].map(name_to_speakers)

# Combine both mappings: prefer mapped_by_code, otherwise mapped_by_name
combined = mapped_by_code.combine_first(mapped_by_name)

# Update 'n_speakers' only where combined is not null
df['n_speakers'] = combined.where(combined.notna(), df['n_speakers'])

In [57]:
df['n_speakers'] = df['n_speakers'].astype(str).str.replace(r'[+,]', '', regex=True)

# Then convert to integers, forcing errors to NaN
df['n_speakers'] = pd.to_numeric(df['n_speakers'], errors='coerce').fillna(0).astype(int)

In [72]:
df.sort_values(by='n_speakers', ascending = False)

,glottocode,language_name,iso_code,language_or_dialect,macroarea,latitude,longitude,dictionary_exists,language_family,bible_exists,...,households_with_phones_pct,households_with_home_internet_pct,individuals_using_internet_pct,avg_download_speed_kbps,avg_upload_speed_kbps,avg_latency_ms,pct_stem_grads,literacy_rate,admin_1_education_level,rnd_gdp_pct
1705,stan1293,English,eng,language,Eurasia,53.000000,-1.000000,1,Indo-European [indo1319],1,...,94.8222,95.8703,95.341211,102414.0,28026.0,25.0,26.31962,100.0,0.930,2.91476
937,mand1415,Mandarin Chinese,cmn,language,Eurasia,40.020900,116.228000,1,Sino-Tibetan [sino1245],1,...,81.4894,80.9409,75.611316,321558.0,72355.0,25.0,0.00000,97.0,0.846,2.43260
4950,stan1318,Standard Arabic,arb,language,Eurasia,27.962500,43.852500,1,Afro-Asiatic [afro1255],1,...,100.0000,100.0000,100.000000,170965.0,96646.0,16.0,21.13605,98.0,0.799,0.46335
1501,stan1288,Spanish,spa,language,Eurasia,40.441400,-1.117880,1,Indo-European [indo1319],1,...,99.1746,96.4472,94.485543,294804.0,246592.0,16.0,22.26665,99.0,0.850,1.42929
2546,hind1269,Hindi,hin,language,Eurasia,25.000000,77.000000,1,Indo-European [indo1319],1,...,56.2472,29.1836,48.075246,36689.0,17709.0,26.0,32.64561,74.0,0.509,0.64636
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5360,urar1245,Urartian,xur,language,Eurasia,38.500000,43.342500,1,Hurro-Urartian [hurr1239],0,...,96.1233,95.5385,85.960742,396332.0,34559.0,9.0,19.44189,97.0,0.783,1.40352
5361,clas1249,Classical-Middle Armenian,xcl,language,Eurasia,38.592316,41.346422,1,Indo-European [indo1319],1,...,96.1233,95.5385,85.960742,29850.0,17928.0,22.0,19.44189,97.0,0.762,1.40352
125,guny1241,Gunya,gyy,language,Australia,-27.064200,145.862000,1,Pama-Nyungan [pama1250],0,...,97.2977,97.3247,94.876188,301628.0,27109.0,8.0,17.39795,100.0,0.917,1.82892
2717,elym1237,Elymian,xly,language,Eurasia,37.941390,12.832220,1,Potentially isolate,0,...,95.4067,83.6821,85.060745,440619.0,216681.0,24.0,24.19916,99.0,0.779,1.45392


In [59]:
df['root_node'] = df['root_node'].replace(r'.*\[[a-z0-9]+\]\[[a-z0-9]{3}\]-l-', 'Isolate', regex=True)
df['root_node'] = df['root_node'].fillna('Potentially isolate')

In [60]:
df = df.drop(columns=['phones'])
df = df.drop(columns=['Unnamed: 0'])

In [61]:
df

,glottocode,language_name,isocodes,level,macroarea,latitude,longitude,dictionary,root_node,bible,...,households with phones,households with home internet,individuals_using_internet,avg_d_kbps,avg_u_kbps,avg_lat_ms,% Stem Grads,literacy rate,educational_attainment_score,RnD % of GDP
0,sout2649,Southern Pashto,pbt,language,Eurasia,31.50000,65.70000,1,Indo-European [indo1319],0,...,49.7956,8.43811,17.600000,2051.0,3197.0,19.0,11.24512,37.000000,0.249,0.1480
1,ormu1247,Ormuri,oru,language,Eurasia,32.68190,69.34440,1,Indo-European [indo1319],0,...,49.7956,8.43811,17.600000,13812.0,2889.0,69.0,11.24512,37.000000,0.383,0.1480
2,haza1239,Hazaragi,haz,language,Eurasia,33.37080,64.51050,1,Indo-European [indo1319],0,...,49.7956,8.43811,17.600000,4204.0,5029.0,261.0,11.24512,37.000000,0.341,0.1480
3,tira1253,Tirahi,tra,language,Eurasia,34.17920,69.59010,1,Indo-European [indo1319],0,...,49.7956,8.43811,17.600000,13812.0,2889.0,69.0,11.24512,37.000000,0.383,0.1480
4,aves1237,Avestan,ave,language,Eurasia,34.21700,62.11200,1,Indo-European [indo1319],0,...,49.7956,8.43811,17.600000,4204.0,5029.0,261.0,11.24512,37.000000,0.341,0.1480
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5998,nort2795,Zimbabwean Ndebele,nde,language,Africa,-20.25130,27.40960,1,Atlantic-Congo [atla1278],1,...,52.0955,52.83320,32.561479,53330.0,4964.0,220.0,30.22403,89.849998,0.542,0.2782
5999,tsoa1238,Northern Tshwa,hio,language,Africa,-19.79740,26.96380,1,Khoe-Kwadi [khoe1240],0,...,52.0955,52.83320,32.561479,106216.0,65104.0,213.0,30.22403,89.849998,0.495,0.2782
6000,many1258,Manyika,mxc,language,Africa,-18.19270,32.80520,1,Atlantic-Congo [atla1278],0,...,52.0955,52.83320,32.561479,102933.0,3280.0,230.0,30.22403,89.849998,0.611,0.2782
6001,shon1251,Shona,sna,language,Africa,-17.20849,29.75725,1,Atlantic-Congo [atla1278],0,...,52.0955,52.83320,32.561479,31278.0,6801.0,322.0,30.22403,89.849998,0.568,0.2782


In [11]:
df.columns

Index(['glottocode', 'language_name', 'isocodes', 'level', 'macroarea',
       'latitude', 'longitude', 'dictionary', 'root_node', 'bible', 'cc_GB',
       'archival_GB', 'datasets_n', 'n_wiki_articles', 'OPUS', 'name_lower',
       'XEUS', 'wikitongues', 'model_n', 'mean_hf_downloads',
       'earliest_datetime', 'wiki_active_users', 'institutional_status',
       'endangerment_status', 'primary_country', 'admin_1_code',
       'country_iso_code', 'countries_where_spoken', 'contested area?',
       'n_speakers', 'geometry', 'nearest_uni', 'second_nearest_uni',
       'distance_to_nearest_km', 'distance_to_second_nearest_km', 'hdi', 'gdp',
       'Cybersecurity legislation/regulation exist', 'households with phones',
       'households with home internet', 'individuals_using_internet',
       'avg_d_kbps', 'avg_u_kbps', 'avg_lat_ms', '% Stem Grads',
       'literacy rate', 'educational_attainment_score', 'RnD % of GDP'],
      dtype='object')

In [62]:
fancy_names = ['Glottocode', 'Language Name', 'ISO code', 'Language or Dialect?', 'Macroarea', 'Latitude', 'Longitude', 'Existence of Dictionary', 'Language Family', 'Existence of a Bible',
             'Gigabytes of Information in Common Crawl','Gigabytes of Information in Archives of Linguistic Documentation', 'Number of Datasets in HuggingFace', 'Number of Articles on Wikipedia', 
              'Gigabytes of Information in Opus', 'Lowercase Name', 'Gigabytes of Information in XEUS', 'Gigabytes of Information in Wikitongues', 'Number of Models in HuggingFace', 
               'Mean Number of Downloads in HuggingFace Models', 'Earliest Date of Upload in HuggingFace Models', 'Number of Active Users in Wikipedia', 'Institutional Status', 
               'Level of Endangerment', 'Main Country where Spoken', 'Code of Administrative Unit 1 (~Region) where Spoken', 'ISO of Main Country where Spoken', 'ISO of Other Countries where Spoken', 
               'Spoken in Contested Area?' 'Number of Speakers', 'Geometry', 'Closest University to the Language', 'Second Closest University to the Language', 'Distance to Closest University (in Km)', 'Distance to Second Closest University (in Km)', 
                'Human Development Index (Aministrative Unit 1 Level)', 'Gross Domestic Product (Administrative Unit 1 Level), Existence of Cybersecurity Legislation/Regulation', 
                   'Percentage of Household with Phones (National Level)', 'Percentage of Households with Home Internet (National Level)', 'Percentage of Individuals Using the Internet (National Level)', 
                      'Average Download Speed (in Kbps - approximately Km² Level)', 'Average Upload Speed (in Kpbs - approximately Km² Level)' , 'Average Latency (in Ms - approximately Km² Level)', 
                      'Percentage of Graduates in Science, Technology, Engineering, and Mathematics (STEM)',
                         'Literacy Rate (National Level)', 'Educational Attainment Level (Aministrative Unit 1 Level)', 'Research and Development Spending as a Percentage of Gross Domestic Product']

In [63]:
software_friendly_names = [
    'glottocode', 'language_name', 'iso_code', 'language_or_dialect', 'macroarea', 'latitude', 'longitude',
    'dictionary_exists', 'language_family', 'bible_exists',
    'cc_gb', 'archival_gb', 'datasets_n', 'wiki_articles_n',
    'opus_gb', 'lowercase_name', 'xeus_gb', 'wikitongues_gb', 'models_n',
    'models_mean_downloads', 'models_earliest_date', 'wiki_active_users', 'institutional_status',
    'endangerment_status', 'primary_country', 'admin_1_code', 'country_iso_code', 'other_country_iso_codes',
    'contested_area', 'n_speakers', 'geometry', 'closest_university', 'second_closest_university',
    'distance_to_closest_university_km', 'distance_to_second_closest_university_km',
    'admin_1_hdi', 'admin_1_gdp', 'cybersecurity_law_exists',
    'households_with_phones_pct', 'households_with_home_internet_pct', 'individuals_using_internet_pct',
    'avg_download_speed_kbps', 'avg_upload_speed_kbps', 'avg_latency_ms','pct_stem_grads',
    'literacy_rate', 'admin_1_education_level', 'rnd_gdp_pct'
]


In [22]:
len(software_friendly_names)

48

In [19]:
len(df.columns)

48

In [64]:
fancy_to_friendly_map = {
    'Glottocode': 'glottocode',
    'Language Name': 'language_name',
    'ISO code': 'iso_code',
    'Language or Dialect?': 'language_or_dialect',
    'Macroarea': 'macroarea',
    'Latitude': 'latitude',
    'Longitude': 'longitude',
    'Existence of Dictionary': 'has_dictionary',
    'Language Family': 'language_family',
    'Existence of a Bible': 'has_bible',
    'Gigabytes of Information in Common Crawl': 'gb_common_crawl',
    'Gigabytes of Information in Archives of Linguistic Documentation': 'gb_archives',
    'Number of Datasets in HuggingFace': 'n_datasets_hf',
    'Number of Articles on Wikipedia': 'n_wikipedia_articles',
    'Gigabytes of Information in Opus': 'gb_opus',
    'Lowercase Name': 'lowercase_name',
    'Gigabytes of Information in XEUS': 'gb_xeus',
    'Gigabytes of Information in Wikitongues': 'gb_wikitongues',
    'Number of Models in HuggingFace': 'n_models_hf',
    'Mean Number of Downloads in HuggingFace Models': 'mean_model_downloads_hf',
    'Earliest Date of Upload in HuggingFace Models': 'earliest_model_upload_hf',
    'Number of Active Users in Wikipedia': 'n_wikipedia_users',
    'Institutional Status': 'institutional_status',
    'Level of Endangerment': 'endangerment_status',
    'Main Country where Spoken': 'primary_country',
    'Code of Administrative Unit 1 (~Region) where Spoken': 'admin_1_code',
    'ISO of Main Country where Spoken': 'country_iso_code',
    'ISO of Other Countries where Spoken': 'other_country_iso_codes',
    'Spoken in Contested Area?Number of Speakers': 'contested_area_speakers',
    'Geometry': 'geometry',
    'Closest University to the Language': 'closest_university',
    'Second Closest University to the Language': 'second_closest_university',
    'Distance to Closest University (in Km)': 'distance_to_closest_uni_km',
    'Distance to Second Closest University (in Km)': 'distance_to_second_closest_uni_km',
    'Human Development Index (Aministrative Unit 1 Level)': 'hdi_admin_1',
    'Gross Domestic Product (Administrative Unit 1 Level), Existence of Cybersecurity Legislation/Regulation': 'gdp_admin_1_cyber_law',
    'Percentage of Household with Phones (National Level)': 'pct_phones',
    'Percentage of Households with Home Internet (National Level)': 'pct_home_internet',
    'Percentage of Individuals Using the Internet (National Level)': 'pct_individuals_internet',
    'Average Download Speed (in Kbps - approximately Km² Level)': 'avg_download_kbps',
    'Average Upload Speed (in Kpbs - approximately Km² Level)': 'avg_upload_kbps',
    'Average Latency (in Ms - approximately Km² Level)': 'avg_latency_ms',
    'Percentage of Graduates in Science, Technology, Engineering, and Mathematics (STEM)': 'pct_stem_grads',
    'Literacy Rate (National Level)': 'literacy_rate',
    'Educational Attainment Level (Aministrative Unit 1 Level)': 'education_attainment_admin_1',
    'Research and Development Spending as a Percentage of Gross Domestic Product': 'rnd_pct_gdp'
}


In [65]:
df.columns = software_friendly_names

In [66]:
df.columns 

Index(['glottocode', 'language_name', 'iso_code', 'language_or_dialect',
       'macroarea', 'latitude', 'longitude', 'dictionary_exists',
       'language_family', 'bible_exists', 'cc_gb', 'archival_gb', 'datasets_n',
       'wiki_articles_n', 'opus_gb', 'lowercase_name', 'xeus_gb',
       'wikitongues_gb', 'models_n', 'models_mean_downloads',
       'models_earliest_date', 'wiki_active_users', 'institutional_status',
       'endangerment_status', 'primary_country', 'admin_1_code',
       'country_iso_code', 'other_country_iso_codes', 'contested_area',
       'n_speakers', 'geometry', 'closest_university',
       'second_closest_university', 'distance_to_closest_university_km',
       'distance_to_second_closest_university_km', 'admin_1_hdi',
       'admin_1_gdp', 'cybersecurity_law_exists', 'households_with_phones_pct',
       'households_with_home_internet_pct', 'individuals_using_internet_pct',
       'avg_download_speed_kbps', 'avg_upload_speed_kbps', 'avg_latency_ms',
       'pct

In [67]:
df['contested_area'] = df['contested_area'].apply(lambda x: 1 if x == 1 else 0)

In [68]:
df

,glottocode,language_name,iso_code,language_or_dialect,macroarea,latitude,longitude,dictionary_exists,language_family,bible_exists,...,households_with_phones_pct,households_with_home_internet_pct,individuals_using_internet_pct,avg_download_speed_kbps,avg_upload_speed_kbps,avg_latency_ms,pct_stem_grads,literacy_rate,admin_1_education_level,rnd_gdp_pct
0,sout2649,Southern Pashto,pbt,language,Eurasia,31.50000,65.70000,1,Indo-European [indo1319],0,...,49.7956,8.43811,17.600000,2051.0,3197.0,19.0,11.24512,37.000000,0.249,0.1480
1,ormu1247,Ormuri,oru,language,Eurasia,32.68190,69.34440,1,Indo-European [indo1319],0,...,49.7956,8.43811,17.600000,13812.0,2889.0,69.0,11.24512,37.000000,0.383,0.1480
2,haza1239,Hazaragi,haz,language,Eurasia,33.37080,64.51050,1,Indo-European [indo1319],0,...,49.7956,8.43811,17.600000,4204.0,5029.0,261.0,11.24512,37.000000,0.341,0.1480
3,tira1253,Tirahi,tra,language,Eurasia,34.17920,69.59010,1,Indo-European [indo1319],0,...,49.7956,8.43811,17.600000,13812.0,2889.0,69.0,11.24512,37.000000,0.383,0.1480
4,aves1237,Avestan,ave,language,Eurasia,34.21700,62.11200,1,Indo-European [indo1319],0,...,49.7956,8.43811,17.600000,4204.0,5029.0,261.0,11.24512,37.000000,0.341,0.1480
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5998,nort2795,Zimbabwean Ndebele,nde,language,Africa,-20.25130,27.40960,1,Atlantic-Congo [atla1278],1,...,52.0955,52.83320,32.561479,53330.0,4964.0,220.0,30.22403,89.849998,0.542,0.2782
5999,tsoa1238,Northern Tshwa,hio,language,Africa,-19.79740,26.96380,1,Khoe-Kwadi [khoe1240],0,...,52.0955,52.83320,32.561479,106216.0,65104.0,213.0,30.22403,89.849998,0.495,0.2782
6000,many1258,Manyika,mxc,language,Africa,-18.19270,32.80520,1,Atlantic-Congo [atla1278],0,...,52.0955,52.83320,32.561479,102933.0,3280.0,230.0,30.22403,89.849998,0.611,0.2782
6001,shon1251,Shona,sna,language,Africa,-17.20849,29.75725,1,Atlantic-Congo [atla1278],0,...,52.0955,52.83320,32.561479,31278.0,6801.0,322.0,30.22403,89.849998,0.568,0.2782


In [71]:
df.loc[df['language_name'] == 'Standard Indonesian', 'n_speakers'] = 252400000
df.loc[df['language_name'] == 'English', 'n_speakers'] = 1350000000

In [73]:
df.to_csv('final_df.csv')